# Data Preprocessing (SPINRec-style Step 1)

目标：将数据快速解析为可训练张量缓存，供后续 MLP/VAE/NCF 统一使用。

In [1]:
%pip install -r ../requirements.txt
%run ./functions.ipynb
print("has seed_everything:", "seed_everything" in globals())
print("has parse_quick_run_data:", "parse_quick_run_data" in globals())
print("has build_model:", "build_model" in globals())

Note: you may need to restart the kernel to use updated packages.
has seed_everything: True
has parse_quick_run_data: True
has build_model: True


In [2]:
# ========== quick_run switch ==========
QUICK_RUN = True
QUICK_DATASET = "Pinterest"  # Pinterest / Yahoo / ML1M

MAX_ROWS = 20000
NUM_BUCKETS = 50000
spinrec_artifacts = None

In [3]:
# Reuse shared utilities.
import importlib.util
import subprocess
import sys
import numpy as np
import pandas as pd

if importlib.util.find_spec("nbformat") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nbformat"])


from pathlib import Path

def _find_root() -> Path:
    c = Path.cwd().resolve()
    for p in [c, *c.parents]:
        if (p / "DATASET").exists() and (p / "recacc").exists():
            return p
    return c

REPO_ROOT = _find_root()
seed_everything(42)


if QUICK_RUN:
    quick_file, quick_sep, quick_has_rating = resolve_quick_run_file(REPO_ROOT, QUICK_DATASET)
    prepared, spinrec_artifacts = parse_quick_run_data(
        data_file=str(quick_file),
        quick_dataset=QUICK_DATASET,
        max_rows=MAX_ROWS,
        num_buckets=NUM_BUCKETS,
        sep=quick_sep,
        has_rating=quick_has_rating,
        seed=42,
        min_items_per_user=2,
        min_users_per_item=2,
        return_artifacts=True,
    )
    NUM_BUCKETS = max(NUM_BUCKETS, int(spinrec_artifacts["required_num_buckets"]))
    DATA_SOURCE = str(quick_file)
else:
    DATA_FILE = str(REPO_ROOT / "DATASET" / "Eleme" / "20220307_01_split" / "washed" / "chunk_aa.csv")
    FIELDS_FILE = str(REPO_ROOT / "recacc" / "eleme_fields.csv")
    prepared = parse_ele_data(
        data_file=DATA_FILE,
        fields_csv=FIELDS_FILE,
        max_rows=MAX_ROWS,
        num_buckets=NUM_BUCKETS,
     )
    DATA_SOURCE = DATA_FILE

print("Repo root:", REPO_ROOT)
print("Quick run mode:", QUICK_RUN, "dataset:", QUICK_DATASET if QUICK_RUN else "Eleme")
print("Data source:", DATA_SOURCE)
print("Loaded samples:", prepared.labels.shape[0])
print("Dense dim:", prepared.dense.shape[1])
print("Sparse fields:", prepared.sparse_fields)
if QUICK_RUN and spinrec_artifacts is not None:
    print("SPINRec-like matrix shape (train/test):", tuple(spinrec_artifacts["train_matrix"].shape), tuple(spinrec_artifacts["test_matrix"].shape))
    print("Users/Items:", spinrec_artifacts["num_users"], spinrec_artifacts["num_items"])

Repo root: H:\TZB
Quick run mode: True dataset: Pinterest
Data source: H:\TZB\recacc\quick_run_data\Pinterest\pinterest_data.csv
Loaded samples: 777818
Dense dim: 1
Sparse fields: ['user_id', 'item_id']
SPINRec-like matrix shape (train/test): (19155, 9366) (19155, 9366)
Users/Items: 19155 9366


In [4]:
cache_path = REPO_ROOT / "recacc" / "log" / "notebook_cache" / "prepared_data.pt"
cache_path.parent.mkdir(parents=True, exist_ok=True)

payload = {
    "dense": prepared.dense,
    "sparse": prepared.sparse,
    "labels": prepared.labels,
    "sample_ids": prepared.sample_ids,
    "sparse_fields": prepared.sparse_fields,
    "dense_fields": prepared.dense_fields,
    "num_buckets": NUM_BUCKETS,
    "quick_run": QUICK_RUN,
    "quick_dataset": QUICK_DATASET,
    "data_source": DATA_SOURCE,
}

if QUICK_RUN and spinrec_artifacts is not None:
    payload.update({
        "spinrec_style": True,
        "spinrec_num_users": spinrec_artifacts["num_users"],
        "spinrec_num_items": spinrec_artifacts["num_items"],
        "spinrec_train_matrix": spinrec_artifacts["train_matrix"],
        "spinrec_test_matrix": spinrec_artifacts["test_matrix"],
        "spinrec_static_test_matrix": spinrec_artifacts["static_test_matrix"],
        "spinrec_static_test_pos": spinrec_artifacts["static_test_pos"],
        "spinrec_static_test_neg": spinrec_artifacts["static_test_neg"],
        "spinrec_pop_array": spinrec_artifacts["pop_array"],
    })

torch.save(payload, cache_path)

print("Saved:", cache_path)
print("Cache keys:", sorted(payload.keys()))

Saved: H:\TZB\recacc\log\notebook_cache\prepared_data.pt
Cache keys: ['data_source', 'dense', 'dense_fields', 'labels', 'num_buckets', 'quick_dataset', 'quick_run', 'sample_ids', 'sparse', 'sparse_fields', 'spinrec_num_items', 'spinrec_num_users', 'spinrec_pop_array', 'spinrec_static_test_matrix', 'spinrec_static_test_neg', 'spinrec_static_test_pos', 'spinrec_style', 'spinrec_test_matrix', 'spinrec_train_matrix']
